# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayazahmedbscsf23-art/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [4]:
import pandas as pd

findings = [
    {
        "Finding": "Logistic Regression - Growth Prediction",
        "Metric": "Holdout Accuracy",
        "Value": "71%",
        "Label Source": "30-day impression change"
    },
    {
        "Finding": "Random Forest - Health Score",
        "Metric": "Feature Importance",
        "Value": "Position: 43%, Impressions: 32%, Scroll Depth: 15%",
        "Label Source": "FlyRank Health Score formula"
    }
]

findings_df = pd.DataFrame(findings)

display(findings_df)

,Finding,Metric,Value,Label Source
0,Logistic Regression - Growth Prediction,Holdout Accuracy,71%,30-day impression change
1,Random Forest - Health Score,Feature Importance,"Position: 43%, Impressions: 32%, Scroll Depth:...",FlyRank Health Score formula


In [10]:
import pandas as pd

df = pd.read_csv("/content/sample_data/california_housing_train.csv")

print("Dataset shape:", df.shape)
print("\nDataset columns:")
print(df.columns.tolist())

display(df.head())

Dataset shape: (17000, 9)

Dataset columns:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


## 2. My model under an honest split (before/after)

## 2. My Model Under an Honest Split (Before/After)

I evaluated my Random Forest Regressor using two different splitting methods.

**Before – Random Split:**
- MAE: 32,359.79
- R² Score: 0.8228

**After – Grouped Split:**
- MAE: 34,645.45
- R² Score: 0.7372

**Comparison and Analysis:**

The random split produced better results than the grouped split.

When I used the grouped split, the MAE increased and the R² score decreased. This indicates that the model performed less effectively when tested on geographically separated groups.

This suggests that the random split may provide an overly optimistic estimate of model performance when geographical similarities exist between training and testing data.

The grouped split provides a more challenging evaluation of the model's ability to predict house prices in unseen geographical groups.

In [11]:
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.base import clone

# Load dataset
df = pd.read_csv(
    "/content/sample_data/california_housing_train.csv"
)

# Input features and target
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]

# Create geographical groups using rounded coordinates
groups = (
    df["latitude"].round(1).astype(str)
    + "_"
    + df["longitude"].round(1).astype(str)
)

# Define model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# --------------------------------
# 1. Random Split
# --------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

random_model = clone(model)
random_model.fit(X_train, y_train)

random_predictions = random_model.predict(X_test)

random_mae = mean_absolute_error(
    y_test, random_predictions
)

random_r2 = r2_score(
    y_test, random_predictions
)

# --------------------------------
# 2. Grouped Split
# --------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

group_model = clone(model)
group_model.fit(X_train_group, y_train_group)

group_predictions = group_model.predict(X_test_group)

group_mae = mean_absolute_error(
    y_test_group, group_predictions
)

group_r2 = r2_score(
    y_test_group, group_predictions
)

# --------------------------------
# 3. Compare Results
# --------------------------------

results = pd.DataFrame({
    "Split Method": [
        "Random Split",
        "Grouped Split"
    ],
    "MAE": [
        random_mae,
        group_mae
    ],
    "R2 Score": [
        random_r2,
        group_r2
    ]
})

display(results)

print("Random Split MAE:", round(random_mae, 2))
print("Grouped Split MAE:", round(group_mae, 2))

print("Random Split R2:", round(random_r2, 4))
print("Grouped Split R2:", round(group_r2, 4))

,Split Method,MAE,R2 Score
0,Random Split,32359.791500,0.822787
1,Grouped Split,34645.446205,0.737165


Random Split MAE: 32359.79
Grouped Split MAE: 34645.45
Random Split R2: 0.8228
Grouped Split R2: 0.7372


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*## 3. Leakage Audit

I performed a leakage audit on my final feature set to check whether any input features contain information from the target variable.

My target variable is `median_house_value`, which represents the house price.

I checked:
- Whether the target column was accidentally included in the input features.
- Whether any feature columns are duplicated.
- Whether the dataset contains duplicate rows.
- Whether any features have an unusually strong correlation with the target.
- Whether the training and testing groups overlap.

The purpose of this audit is to identify possible data leakage and make the model evaluation more reliable.

In [12]:
import pandas as pd

# Load the dataset
df = pd.read_csv(
    "/content/sample_data/california_housing_train.csv"
)

# Define final features and target
target = "median_house_value"

X = df.drop(columns=[target])
y = df[target]

print("========== LEAKAGE AUDIT ==========")

# 1. Check whether target is included in features
print("\n1. Target Column Check")

if target in X.columns:
    print("WARNING: Target leakage detected!")
else:
    print("PASS: Target column is not included in features.")

# 2. Check duplicate feature columns
print("\n2. Duplicate Feature Columns")

duplicate_columns = X.columns[X.columns.duplicated()].tolist()

if duplicate_columns:
    print("WARNING: Duplicate columns:", duplicate_columns)
else:
    print("PASS: No duplicate feature columns.")

# 3. Check duplicate rows
print("\n3. Duplicate Rows")

duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

# 4. Check feature correlation with target
print("\n4. Feature Correlation With Target")

correlations = X.corrwith(y).abs().sort_values(ascending=False)

display(correlations.to_frame("Absolute Correlation"))

# 5. Check exact duplicate feature values
print("\n5. Identical Feature Columns")

identical_columns = []

columns = X.columns.tolist()

for i in range(len(columns)):
    for j in range(i + 1, len(columns)):
        if X[columns[i]].equals(X[columns[j]]):
            identical_columns.append((columns[i], columns[j]))

if identical_columns:
    print("WARNING: Identical columns found:", identical_columns)
else:
    print("PASS: No identical feature columns.")

# 6. Check geographical group overlap
print("\n6. Group Overlap Check")

groups = (
    df["latitude"].round(1).astype(str)
    + "_"
    + df["longitude"].round(1).astype(str)
)

train_groups = set(groups.iloc[train_idx])
test_groups = set(groups.iloc[test_idx])

overlap = train_groups.intersection(test_groups)

print("Overlapping groups:", len(overlap))

if len(overlap) == 0:
    print("PASS: No geographical groups overlap.")
else:
    print("WARNING: Some geographical groups overlap.")

print("\n========== AUDIT COMPLETED ==========")

========== LEAKAGE AUDIT ==========

1. Target Column Check
PASS: Target column is not included in features.

2. Duplicate Feature Columns
PASS: No duplicate feature columns.

3. Duplicate Rows
Duplicate rows: 0

4. Feature Correlation With Target


,Absolute Correlation
median_income,0.691871
latitude,0.144917
total_rooms,0.130991
housing_median_age,0.106758
households,0.061031
total_bedrooms,0.045783
longitude,0.044982
population,0.027850



5. Identical Feature Columns
PASS: No identical feature columns.

6. Group Overlap Check
Overlapping groups: 0
PASS: No geographical groups overlap.

========== AUDIT COMPLETED ==========


## 4. Claim rewrite

## 4. Claim Rewrite

### Original Bold Claim

"My Random Forest model accurately predicts house prices and performs reliably on unseen geographical areas."

### Rewritten Safe Claim

"In my experiment, the Random Forest model achieved an R² score of 0.8228 with a random split and 0.7372 with a geographical grouped split. The results show that model performance decreased when evaluated on geographically separated groups. This suggests that the model may perform less effectively on unseen geographical groups. Therefore, the model should be considered a decision-support and estimation tool rather than a guaranteed house-price prediction system."

### Conclusion

The results describe the performance observed in my dataset and evaluation setup. They do not guarantee the same performance in every geographical area.

In [13]:
original_claim = (
    "My Random Forest model accurately predicts house prices "
    "and performs reliably on unseen geographical areas."
)

rewritten_claim = (
    "In my experiment, the model achieved an R2 score of 0.8228 "
    "with a random split and 0.7372 with a geographical grouped split. "
    "Performance decreased on geographically separated groups. "
    "The model may be useful as a house-price estimation "
    "and decision-support tool, but its predictions are not guaranteed."
)

print("Original Claim:\n", original_claim)

print("\nRewritten Safe Claim:\n", rewritten_claim)

Original Claim:
 My Random Forest model accurately predicts house prices and performs reliably on unseen geographical areas.

Rewritten Safe Claim:
 In my experiment, the model achieved an R2 score of 0.8228 with a random split and 0.7372 with a geographical grouped split. Performance decreased on geographically separated groups. The model may be useful as a house-price estimation and decision-support tool, but its predictions are not guaranteed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.